# Lesson 3.3 — Client-Side vs Server-Side Tools and Approval Patterns

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish client-side and server-side tool execution
- Implement an approval gate for sensitive/destructive actions
- Classify tools as read-only vs mutating to drive policy
- Apply least privilege to the agentic harness dispatch

**You will produce:** An approval-gated dispatcher that blocks mutating tools until confirmed.

## Concepts

Tools differ by **where** they run and **how risky** they are:

- **Client-side tools**: executed by your application code (full control over auth, logging, and approval). Most custom tools are client-side.
- **Server-side / built-in tools**: executed by Anthropic's infrastructure (e.g., certain built-in capabilities) - convenient but with less local control.

**Approval patterns**: for destructive or sensitive actions (delete, pay, email), gate execution behind explicit confirmation - a human-in-the-loop prompt, an allowlist, or a policy check. Combine with **least privilege**: the harness should only expose the tools a task truly needs. This directly supports Domain 7 security.

> **Reference:** [Anthropic - Tool use (how it works)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Classify tools by risk

Tag each tool read-only or mutating to drive the approval policy.

In [ ]:
TOOL_RISK = {
    "search_orders": "read",
    "cancel_booking": "mutate",
    "issue_refund": "mutate",
}

def needs_approval(tool_name):
    return TOOL_RISK.get(tool_name, "mutate") == "mutate"  # default-deny

print(needs_approval("search_orders"), needs_approval("issue_refund"))

## Code-Along: Step 2 — Approval-gated dispatcher

Block mutating tools until an approval callback returns True.

In [ ]:
def dispatch(tool_name, tool_input, approve=lambda n, i: False):
    if needs_approval(tool_name) and not approve(tool_name, tool_input):
        return {"ok": False, "error": f"approval required for '{tool_name}'"}
    # ... execute the real tool here ...
    return {"ok": True, "result": f"executed {tool_name} with {tool_input}"}

# Auto-deny (default): mutating tool blocked
print(dispatch("issue_refund", {"amount": 50}))
# Explicit human approval callback
print(dispatch("issue_refund", {"amount": 50}, approve=lambda n, i: True))

## Exercise

Write `build_toolset(task)` that returns only the tools a given task needs (least privilege). For a 'read-only reporting' task it must exclude all mutating tools.

In [ ]:
ALL_TOOLS = ["search_orders", "cancel_booking", "issue_refund"]
def build_toolset(task):
    # TODO: if task == 'reporting', return only read tools
    # TODO: otherwise return the minimal set the task requires
    pass

## Key Takeaways

- Client-side tools run in your code; server-side/built-in tools run on Anthropic infra.
- Gate destructive/sensitive actions behind explicit approval.
- Classify tools read vs mutate and default-deny unknowns.
- Apply least privilege - expose only the tools a task needs.

---

**Next:** [Lesson 3.4 — MCP Server Development: Resources, Tools, and Prompts](./lesson-3.4-mcp-server-development.ipynb)